In [0]:
# ============================================================
# 01_JOB_DEMAND — SELF-CONTAINED VERSION
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# 1. READ CLEAN SILVER
# ============================================================

df = spark.table("job_market.silver.jobs_clean")

total_jobs = df.count()

print("Clean Silver records:", total_jobs)


# ============================================================
# 2. PREPARE JOB TITLE FIELDS
# ============================================================

job_demand_base = (
    df
    .withColumn(
        "job_title_display",
        F.trim(
            F.regexp_replace(
                F.col("job_title"),
                r"\s+",
                " "
            )
        )
    )
    .withColumn(
        "job_title_normalized",
        F.lower(
            F.trim(
                F.regexp_replace(
                    F.col("job_title"),
                    r"\s+",
                    " "
                )
            )
        )
    )
)


# ============================================================
# 3. AGGREGATE JOB DEMAND
# ============================================================

job_demand = (
    job_demand_base
    .groupBy(
        "job_title_normalized"
    )
    .agg(

        F.count("*").alias("job_count"),

        F.countDistinct(
            "company"
        ).alias("company_count"),

        F.countDistinct(
            "location_normalized"
        ).alias("location_count"),

        F.sum(
            F.when(
                F.col("is_remote") == True,
                1
            ).otherwise(0)
        ).alias("remote_job_count"),

        F.sum(
            F.when(
                F.lower(
                    F.col("workplace_type")
                ) == "hybrid",
                1
            ).otherwise(0)
        ).alias("hybrid_job_count"),

        F.min(
            "posted_at"
        ).alias("earliest_posted_at"),

        F.max(
            "posted_at"
        ).alias("latest_posted_at")
    )
)


# ============================================================
# 4. ADD DISPLAY JOB TITLE
# ============================================================

title_lookup = (
    job_demand_base
    .groupBy(
        "job_title_normalized"
    )
    .agg(
        F.min(
            "job_title_display"
        ).alias("job_title")
    )
)

job_demand = (
    job_demand
    .join(
        title_lookup,
        on="job_title_normalized",
        how="left"
    )
)


# ============================================================
# 5. CALCULATE DEMAND METRICS
# ============================================================

job_demand = (
    job_demand

    .withColumn(
        "demand_percentage",
        F.round(
            (
                F.col("job_count") /
                F.lit(total_jobs)
            ) * 100,
            2
        )
    )

    .withColumn(
        "remote_percentage",
        F.round(
            (
                F.col("remote_job_count") /
                F.col("job_count")
            ) * 100,
            2
        )
    )

    .withColumn(
        "hybrid_percentage",
        F.round(
            (
                F.col("hybrid_job_count") /
                F.col("job_count")
            ) * 100,
            2
        )
    )
)


# ============================================================
# 6. RANK JOB TITLES
# ============================================================

ranking_window = Window.orderBy(
    F.desc("job_count"),
    F.asc("job_title_normalized")
)

job_demand = (
    job_demand
    .withColumn(
        "job_demand_rank",
        F.dense_rank().over(
            ranking_window
        )
    )
)


# ============================================================
# 7. FINAL GOLD SCHEMA
# ============================================================

job_demand = (
    job_demand
    .select(
        "job_demand_rank",
        "job_title",
        "job_title_normalized",
        "job_count",
        "demand_percentage",
        "company_count",
        "location_count",
        "remote_job_count",
        "remote_percentage",
        "hybrid_job_count",
        "hybrid_percentage",
        "earliest_posted_at",
        "latest_posted_at"
    )
    .orderBy(
        F.asc("job_demand_rank")
    )
)


# ============================================================
# 8. VALIDATE
# ============================================================

gold_job_count = (
    job_demand
    .agg(
        F.sum(
            "job_count"
        ).alias(
            "total_job_count"
        )
    )
    .collect()[0]["total_job_count"]
)

print("======================================")
print("JOB DEMAND VALIDATION")
print("======================================")
print("Silver records :", total_jobs)
print("Gold job count :", gold_job_count)
print(
    "Difference     :",
    total_jobs - gold_job_count
)
print("======================================")

assert total_jobs == gold_job_count, \
    "Job demand aggregation does not reconcile!"

print("✓ Job demand reconciliation passed.")


# ============================================================
# 9. SAVE GOLD TABLE
# ============================================================

job_demand.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "job_market.gold.job_demand"
    )

print("✓ Created:")
print("job_market.gold.job_demand")


# ============================================================
# 10. DISPLAY RESULT
# ============================================================

display(
    job_demand.limit(30)
)

Clean Silver records: 2221


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


JOB DEMAND VALIDATION
Silver records : 2221
Gold job count : 2221
Difference     : 0
✓ Job demand reconciliation passed.
✓ Created:
job_market.gold.job_demand


job_demand_rank,job_title,job_title_normalized,job_count,demand_percentage,company_count,location_count,remote_job_count,remote_percentage,hybrid_job_count,hybrid_percentage,earliest_posted_at,latest_posted_at
1,"Software Engineer, New Grad","software engineer, new grad",10,0.45,2,10,0,0.0,4,40.0,2020-10-27T18:32:40.000Z,2026-09-03T03:28:52.000Z
2,Deployment Strategist,deployment strategist,9,0.41,1,9,0,0.0,7,77.78,2013-03-08T00:00:00.000Z,2026-06-25T11:39:38.000Z
3,Forward Deployed Software Engineer,forward deployed software engineer,9,0.41,1,9,0,0.0,7,77.78,2009-12-05T00:00:00.000Z,2024-05-01T17:20:33.000Z
4,Deployment Strategist - US Government,deployment strategist - us government,6,0.27,1,6,0,0.0,1,16.67,2020-05-09T00:52:07.000Z,2026-09-24T14:53:54.000Z
5,"Deployment Strategist, Build to Apply - US Government","deployment strategist, build to apply - us government",6,0.27,1,6,0,0.0,0,0.0,2026-02-04T22:53:06.000Z,2026-02-11T17:06:56.000Z
6,Forward Deployed Software Engineer - US Government,forward deployed software engineer - us government,6,0.27,1,6,0,0.0,1,16.67,2020-04-15T21:27:49.000Z,2026-09-24T14:50:21.000Z
7,"Software Engineer, Internship","software engineer, internship",6,0.27,1,6,0,0.0,0,0.0,2021-07-01T21:52:15.000Z,2026-01-14T15:28:44.000Z
8,Senior Forward Deployed Engineer,senior forward deployed engineer,5,0.23,1,1,0,0.0,0,0.0,2026-08-05T21:51:40.000Z,2026-09-21T22:04:04.000Z
9,Software Engineer,software engineer,5,0.23,3,5,1,20.0,0,0.0,2026-06-17T13:41:58.000Z,2026-10-02T18:52:10.000Z
10,Concierge Specialist IV,concierge specialist iv,4,0.18,1,2,0,0.0,0,0.0,2026-06-11T00:35:39.000Z,2026-06-26T21:03:00.000Z


In [0]:
# ============================================================
# 01_JOB_DEMAND — SELF-CONTAINED VERSION
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# 1. READ CLEAN SILVER
# ============================================================

df = spark.table("job_market.silver.jobs_clean")

total_jobs = df.count()

print("Clean Silver records:", total_jobs)


# ============================================================
# 2. PREPARE JOB TITLE FIELDS
# ============================================================

job_demand_base = (
    df
    .withColumn(
        "job_title_display",
        F.trim(
            F.regexp_replace(
                F.col("job_title"),
                r"\s+",
                " "
            )
        )
    )
    .withColumn(
        "job_title_normalized",
        F.lower(
            F.trim(
                F.regexp_replace(
                    F.col("job_title"),
                    r"\s+",
                    " "
                )
            )
        )
    )
)


# ============================================================
# 3. AGGREGATE JOB DEMAND
# ============================================================

job_demand = (
    job_demand_base
    .groupBy(
        "job_title_normalized"
    )
    .agg(

        F.count("*").alias("job_count"),

        F.countDistinct(
            "company"
        ).alias("company_count"),

        F.countDistinct(
            "location_normalized"
        ).alias("location_count"),

        F.sum(
            F.when(
                F.col("is_remote") == True,
                1
            ).otherwise(0)
        ).alias("remote_job_count"),

        F.sum(
            F.when(
                F.lower(
                    F.col("workplace_type")
                ) == "hybrid",
                1
            ).otherwise(0)
        ).alias("hybrid_job_count"),

        F.min(
            "posted_at"
        ).alias("earliest_posted_at"),

        F.max(
            "posted_at"
        ).alias("latest_posted_at")
    )
)


# ============================================================
# 4. ADD DISPLAY JOB TITLE
# ============================================================

title_lookup = (
    job_demand_base
    .groupBy(
        "job_title_normalized"
    )
    .agg(
        F.min(
            "job_title_display"
        ).alias("job_title")
    )
)

job_demand = (
    job_demand
    .join(
        title_lookup,
        on="job_title_normalized",
        how="left"
    )
)


# ============================================================
# 5. CALCULATE DEMAND METRICS
# ============================================================

job_demand = (
    job_demand

    .withColumn(
        "demand_percentage",
        F.round(
            (
                F.col("job_count") /
                F.lit(total_jobs)
            ) * 100,
            2
        )
    )

    .withColumn(
        "remote_percentage",
        F.round(
            (
                F.col("remote_job_count") /
                F.col("job_count")
            ) * 100,
            2
        )
    )

    .withColumn(
        "hybrid_percentage",
        F.round(
            (
                F.col("hybrid_job_count") /
                F.col("job_count")
            ) * 100,
            2
        )
    )
)


# ============================================================
# 6. RANK JOB TITLES
# ============================================================

ranking_window = Window.orderBy(
    F.desc("job_count"),
    F.asc("job_title_normalized")
)

job_demand = (
    job_demand
    .withColumn(
        "job_demand_rank",
        F.dense_rank().over(
            ranking_window
        )
    )
)


# ============================================================
# 7. FINAL GOLD SCHEMA
# ============================================================

job_demand = (
    job_demand
    .select(
        "job_demand_rank",
        "job_title",
        "job_title_normalized",
        "job_count",
        "demand_percentage",
        "company_count",
        "location_count",
        "remote_job_count",
        "remote_percentage",
        "hybrid_job_count",
        "hybrid_percentage",
        "earliest_posted_at",
        "latest_posted_at"
    )
    .orderBy(
        F.asc("job_demand_rank")
    )
)


# ============================================================
# 8. VALIDATE
# ============================================================

gold_job_count = (
    job_demand
    .agg(
        F.sum(
            "job_count"
        ).alias(
            "total_job_count"
        )
    )
    .collect()[0]["total_job_count"]
)

print("======================================")
print("JOB DEMAND VALIDATION")
print("======================================")
print("Silver records :", total_jobs)
print("Gold job count :", gold_job_count)
print(
    "Difference     :",
    total_jobs - gold_job_count
)
print("======================================")

assert total_jobs == gold_job_count, \
    "Job demand aggregation does not reconcile!"

print("✓ Job demand reconciliation passed.")


# ============================================================
# 9. SAVE GOLD TABLE
# ============================================================

job_demand.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "job_market.gold.job_demand"
    )

print("✓ Created:")
print("job_market.gold.job_demand")


# ============================================================
# 10. DISPLAY RESULT
# ============================================================

display(
    job_demand.limit(30)
)

Clean Silver records: 2221


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


JOB DEMAND VALIDATION
Silver records : 2221
Gold job count : 2221
Difference     : 0
✓ Job demand reconciliation passed.
✓ Created:
job_market.gold.job_demand


job_demand_rank,job_title,job_title_normalized,job_count,demand_percentage,company_count,location_count,remote_job_count,remote_percentage,hybrid_job_count,hybrid_percentage,earliest_posted_at,latest_posted_at
1,"Software Engineer, New Grad","software engineer, new grad",10,0.45,2,10,0,0.0,4,40.0,2020-10-27T18:32:40.000Z,2026-09-03T03:28:52.000Z
2,Deployment Strategist,deployment strategist,9,0.41,1,9,0,0.0,7,77.78,2013-03-08T00:00:00.000Z,2026-06-25T11:39:38.000Z
3,Forward Deployed Software Engineer,forward deployed software engineer,9,0.41,1,9,0,0.0,7,77.78,2009-12-05T00:00:00.000Z,2024-05-01T17:20:33.000Z
4,Deployment Strategist - US Government,deployment strategist - us government,6,0.27,1,6,0,0.0,1,16.67,2020-05-09T00:52:07.000Z,2026-09-24T14:53:54.000Z
5,"Deployment Strategist, Build to Apply - US Government","deployment strategist, build to apply - us government",6,0.27,1,6,0,0.0,0,0.0,2026-02-04T22:53:06.000Z,2026-02-11T17:06:56.000Z
6,Forward Deployed Software Engineer - US Government,forward deployed software engineer - us government,6,0.27,1,6,0,0.0,1,16.67,2020-04-15T21:27:49.000Z,2026-09-24T14:50:21.000Z
7,"Software Engineer, Internship","software engineer, internship",6,0.27,1,6,0,0.0,0,0.0,2021-07-01T21:52:15.000Z,2026-01-14T15:28:44.000Z
8,Senior Forward Deployed Engineer,senior forward deployed engineer,5,0.23,1,1,0,0.0,0,0.0,2026-08-05T21:51:40.000Z,2026-09-21T22:04:04.000Z
9,Software Engineer,software engineer,5,0.23,3,5,1,20.0,0,0.0,2026-06-17T13:41:58.000Z,2026-10-02T18:52:10.000Z
10,Concierge Specialist IV,concierge specialist iv,4,0.18,1,2,0,0.0,0,0.0,2026-06-11T00:35:39.000Z,2026-06-26T21:03:00.000Z
